In [ ]:
import pandas as pd
import datetime as dt
import numpy as np 
import glob
import matplotlib.pyplot as plt 

from matplotlib.ticker import AutoMinorLocator

from JunoPy.spice import InitSpiceKernels, get_juno_position
from JunoPy.mag import MagData
from JunoPy.library import convert_pos_spherical

In [ ]:
def get_juno_positions():
    start = dt.datetime(2016, 6, 20, 17, 30)
    end   = dt.datetime(2020, 5, 7, 23)

    time_array = pd.date_range(
            start=start, end=end, freq='H').to_pydatetime().tolist()

    InitSpiceKernels()
    position, _ = get_juno_position(time_array)
    
    return position, time_array

position, time_array = get_juno_positions()

In [ ]:
def calc_coeff_bs(p):
    A = -1.107 + 1.591 * p**(-0.25)
    B = -0.566 - 0.812 * p**(-0.25)
    C =  0.048 - 0.059 * p**(-0.25)
    D =  0.077 - 0.038 * p
    E = -0.874 - 0.299 * p
    F = -0.055 + 0.124 * p
    return A, B, C, D, E, F
    
def calc_coeff_mp(p):
    A = -0.134 + 0.488 * p**(-0.25)
    B = -0.581 - 0.225 * p**(-0.25)
    C = -0.186 - 0.016 * p**(-0.25)
    D = -0.014 + 0.096 * p
    E = -0.814 - 0.811 * p
    F = -0.050 + 0.168 * p
    return A, B, C, D, E, F

def calc_equatorial(p, N, ctype='MP'):
    
    if 'MP' in ctype:
        A, B, C, D, E, F = calc_coeff_mp(p)
    else:
        A, B, C, D, E, F = calc_coeff_bs(p)
        
    phi = np.linspace(-150, 150, N) * np.pi / 180
    
    a = C * np.cos(phi)**2 + E * np.sin(phi)**2 + F * np.sin(phi) * np.cos(phi)
    b = B * np.cos(phi) + D * np.sin(phi)
    c = A
    
    root1 = (-b + np.sqrt(b**2 - 4 * a * c)) / (2 * a)
    root2 = (-b - np.sqrt(b**2 - 4 * a * c)) / (2 * a)
    
    r = np.zeros(N)
    for i in range(N):
        if root1[i] > 0:
            r[i] = root1[i]
        else:
            r[i] = root2[i]
        
    x = r * 120 * np.cos(phi)
    y = r * 120 * np.sin(phi)    
    

    return x, y
    

x_MP, y_MP = calc_equatorial(0.383, 101, ctype='MP')
x_BS, y_BS = calc_equatorial(0.382, 101, ctype='BS')

In [ ]:
event_list = pd.read_csv('auto_plasmoid_list_20210525_window10.csv', skipinitialspace=True)
event_list['Start'] = pd.to_datetime(event_list['Start'], format='%Y-%jT%H:%M:%S') 
event_list['End'] = pd.to_datetime(event_list['End'], format='%Y-%jT%H:%M:%S') 
event_list['Durations'] = event_list.apply(lambda x: (x['End'] - x['Start']).total_seconds(), axis=1)
event_list

# Get eigenvectors for all the data.

In [ ]:
def read_and_crop(magfile, start, end):
    """Read and crop magnetometer data."""
    magdata = MagData(filename=magfile)
    start_cut = start - dt.timedelta(minutes=10)
    end_cut   = end   + dt.timedelta(minutes=10)
    magdata.df = magdata.df.loc[start_cut:end_cut]

    magdata_mva = MagData(filename=magfile)
    magdata_mva.df = magdata_mva.df.loc[start:end].copy()
    magdata_mva.perform_mva()

    return magdata, magdata_mva

def get_eigenvectors(event_list):
    x1 = []
    x2 = []
    x3 = []
    for row in event_list.itertuples():
        magfile = glob.glob('data/mag/*{:s}ss*.hdf5'.format(row[1].strftime('%Y%j')))[0]
        magdata, magdata_mva = read_and_crop(magfile, row[1], row[2])
        
        x1.append(magdata_mva.x1)
        x2.append(magdata_mva.x2)
        x3.append(magdata_mva.x3)

    x1 = np.asarray(x1)
    x2 = np.asarray(x2)
    x3 = np.asarray(x3)
    
    for i in range(len(x3)):
        if x3[i, 2] < 0:
            x1[i, :] = x1[i, :] * -1
            x2[i, :] = x2[i, :] * -1
            x3[i, :] = x3[i, :] * -1

    return x1, x2, x3

x1, x2, x3 = get_eigenvectors(event_list)

# 1. Plot eigenvectors colored by duration.

In [ ]:
def plot_eigenvector_duration(event_list, x1, x2, x3):
    gs_kw = dict(height_ratios=[130, 65], hspace=0)
    fig, ax = plt.subplots(2, 1, sharex=True, dpi=300, gridspec_kw=gs_kw, figsize=(6, 8))

    ax1 = ax[0]
    ax2 = ax[1]
    
    position_mva, _ = get_juno_position(event_list['Start'])
    
    ax1.plot(position[:, 0], position[:, 1], color=[0.3, 0.3, 0.3], zorder=1, alpha=0.5)
    ax2.plot(position[:, 0], position[:, 2], color=[0.3, 0.3, 0.3], zorder=1, alpha=0.5)
    # ax1.plot(position_masked[:, 0], position_masked[:, 1], color=[0.3, 0.3, 0.3], zorder=1)
    # ax2.plot(position_masked[:, 0], position_masked[:, 2], color=[0.3, 0.3, 0.3], zorder=1)

    ax1.plot(x_BS, y_BS, color='k', zorder=0)
    ax1.plot(x_MP, y_MP, color='k', ls='dashed', zorder=0)

    ax1.xaxis.set_minor_locator(AutoMinorLocator())
    ax1.yaxis.set_minor_locator(AutoMinorLocator())
    ax2.xaxis.set_minor_locator(AutoMinorLocator())
    ax2.yaxis.set_minor_locator(AutoMinorLocator())
    ax1.grid('on', which='minor', lw=0.5, ls='dotted')
    ax1.grid(color='k', ls='dashed', lw=0.25)
    ax2.grid('on', which='minor', lw=0.5, ls='dotted')
    ax2.grid(color='k', ls='dashed', lw=0.25)

    ax1.set_xlabel('X [R]', weight='bold')
    ax1.set_ylabel('Y [R]', weight='bold')
    ax2.set_xlabel('X [R]', weight='bold')
    ax2.set_ylabel('Z [R]', weight='bold')



    # ax1.set_xlim([-105, 40])
    ax1.set_ylim([-120, 10])

    ax2.set_xlim([-105, 40])
    ax2.set_ylim([-45, 20])

    # ax1.set_adjustable("datalim", share=True)

    ax1.set_aspect('equal', share=True)
    ax2.set_aspect('equal', share=True)

    sc1 = ax1.scatter(position_mva[:, 0], position_mva[:, 1], c=event_list['Durations'], zorder=2, marker='o', cmap='jet')
    sc2 = ax2.scatter(position_mva[:, 0], position_mva[:, 2], c=event_list['Durations'], zorder=2, marker='o', cmap='jet')

    q1 = ax1.quiver(position_mva[:, 0], position_mva[:, 1], x2[:, 0], x2[:, 1])
    q2 = ax2.quiver(position_mva[:, 0], position_mva[:, 2], x3[:, 0], x3[:, 2])

    bbox2 = ax2.get_position()
    bbox1 = ax1.get_position()
    cax = fig.add_axes([bbox2.x1, bbox2.y0, 0.016, bbox1.y1- bbox2.y0])


    ax1.arrow(20, -115, 10, 0, color='k', head_width=3)
    ax1.text(20, -110, 'To Sun')
    ax1.text(30, -60, 'MP')
    ax1.text(30, -90, 'BS')

    ax1.text(-100, 2, 'Vectors of $x_M$', weight='bold')
    ax2.text(-100, 2, 'Vectors of $x_N$', weight='bold')
    ax2.text(-60, -40, 'Juno Trajectory', color=[0.3, 0.3, 0.3], alpha=0.5, weight='bold')

    ax1.add_artist(plt.Circle((0, 0), 1.0, color='k', zorder=0))
    ax2.add_artist(plt.Circle((0, 0), 1.0, color='k', zorder=0))

    cbar2 = fig.colorbar(sc2, cax=cax);
    cbar2.set_label('Peak-to-peak Duration (s)')
    plt.subplots_adjust(wspace=0.3)
    fig.savefig('out_images/TrajectoryLocationofEvent_quiver_selected.png')
    
plot_eigenvector_duration(event_list, x1, x2, x3)

In [ ]:
# 2. Plot information about north versus south., motion of plasmoid

def plot_motion_events(event_list, x1, x2, x3):
    gs_kw = dict(height_ratios=[150, 75], hspace=0)
    fig, ax = plt.subplots(2, 1, sharex=True, dpi=300, gridspec_kw=gs_kw, figsize=(6, 8))

    ax1 = ax[0]
    ax2 = ax[1]
    
#     ax1.plot(position[:, 0], position[:, 1], color=[0.3, 0.3, 0.3], zorder=1, alpha=0.5)
#     ax2.plot(position[:, 0], position[:, 2], color=[0.3, 0.3, 0.3], zorder=1, alpha=0.5)
    # ax1.plot(position_masked[:, 0], position_masked[:, 1], color=[0.3, 0.3, 0.3], zorder=1)
    # ax2.plot(position_masked[:, 0], position_masked[:, 2], color=[0.3, 0.3, 0.3], zorder=1)

    ax1.plot(x_BS, y_BS, color='k', zorder=0)
    ax1.plot(x_MP, y_MP, color='k', ls='dashed', zorder=0)

    ax1.xaxis.set_minor_locator(AutoMinorLocator())
    ax1.yaxis.set_minor_locator(AutoMinorLocator())
    ax2.xaxis.set_minor_locator(AutoMinorLocator())
    ax2.yaxis.set_minor_locator(AutoMinorLocator())
    ax1.grid('on', which='minor', lw=0.5, ls='dotted')
    ax1.grid(color='k', ls='dashed', lw=0.25)
    ax2.grid('on', which='minor', lw=0.5, ls='dotted')
    ax2.grid(color='k', ls='dashed', lw=0.25)

    ax1.set_xlabel('X [R]', weight='bold')
    ax1.set_ylabel('Y [R]', weight='bold')
    ax2.set_xlabel('X [R]', weight='bold')
    ax2.set_ylabel('Z [R]', weight='bold')



    # ax1.set_xlim([-105, 40])
    ax1.set_ylim([-120, 30])

    ax2.set_xlim([-105, 40])
    ax2.set_ylim([-55, 20])

    # ax1.set_adjustable("datalim", share=True)

    ax1.set_aspect('equal', share=True)
    ax2.set_aspect('equal', share=True)
    
    events_tailward = event_list[event_list['Motion'] == 'tailward']
    events_planetward = event_list[event_list['Motion'] == 'planetward']
    
    positions_tailward, _ = get_juno_position(events_tailward['Start'])
    positions_planetward, _ = get_juno_position(events_planetward['Start'])
    
    sc1 = ax1.scatter(positions_tailward[:, 0], positions_tailward[:, 1], 
                      c='r', zorder=2, marker='o', s=20, label='Positive-to-Negative (S to N)')
    
    sc2 = ax1.scatter(positions_planetward[:, 0], positions_planetward[:, 1], 
                      c='b', zorder=2, marker='o', s=20, label='Negative-to-Positive (N to S)')
    
    sc3 = ax2.scatter(positions_tailward[:, 0], positions_tailward[:, 2], c='r', zorder=2, marker='o', s=20, label='tailward')
    sc4 = ax2.scatter(positions_planetward[:, 0], positions_planetward[:, 2], c='b', zorder=2, marker='o', s=20, label='planetward')
        
    # q1 = ax1.quiver(position_mva[:, 0], position_mva[:, 1], x2[:, 0], x2[:, 1])
    # q2 = ax2.quiver(position_mva[:, 0], position_mva[:, 2], x3[:, 0], x3[:, 2])

#     bbox2 = ax2.get_position()
#     bbox1 = ax1.get_position()
#     cax = fig.add_axes([bbox2.x1, bbox2.y0, 0.016, bbox1.y1- bbox2.y0])

    ax1.legend(loc='upper right')
    
    ax1.arrow(20, -115, 10, 0, color='k', head_width=3)
    ax1.text(20, -110, 'To Sun')
    ax1.text(30, -60, 'MP')
    ax1.text(30, -90, 'BS')
    ax1.text(2, 1, 'Jupiter')
    
    ax1.text(0.1, 0.9, 'a)', bbox=dict(facecolor='w', edgecolor='k'), transform=ax1.transAxes)
    ax2.text(0.1, 0.9, 'b)', bbox=dict(facecolor='w', edgecolor='k'), transform=ax2.transAxes)
    
    # ax2.text(-60, -40, 'Juno Trajectory', color=[0.3, 0.3, 0.3], alpha=0.5, weight='bold')

    ax1.add_artist(plt.Circle((0, 0), 1.0, color='k', zorder=0))
    ax2.add_artist(plt.Circle((0, 0), 1.0, color='k', zorder=0))

#     cbar2 = fig.colorbar(sc2, cax=cax);
#     cbar2.set_label('Peak-to-peak Duration (s)')
    plt.subplots_adjust(wspace=0.3)
    fig.savefig('out_images/TrajectoryLocationofEvent_motion.png', bbox_inches='tight')
    
plot_motion_events(event_list, x1, x2, x3)

In [ ]:
# Histogram of event location

def plot_histograms():
    R = np.sqrt(np.sum(position**2, axis=1))
    LT = 6 / 90 * np.arctan2(position[:, 1], position[:, 0]) * 180 / np.pi + 12


    R_occurence = np.histogram(R, bins=np.linspace(0, 100, 11))
    LT_occurence = np.histogram(LT, bins=np.linspace(0, 24, 25))


    events_tailward = event_list[event_list['Motion'] == 'tailward']
    events_planetward = event_list[event_list['Motion'] == 'planetward']

    positions_tailward, _ = get_juno_position(events_tailward['Start'])
    positions_planetward, _ = get_juno_position(events_planetward['Start'])


    fig, ax = plt.subplots(3, 2, dpi=300, figsize=(5, 5)) #, gridspec_kw=dict(hspace=0.4, wspace=0.3))

    R_tailward = np.sqrt(np.sum(positions_tailward**2, axis=1))
    LT_tailward = 6 / 90 * np.arctan2(positions_tailward[:, 1], positions_tailward[:, 0]) * 180 / np.pi + 12

    R_planetward = np.sqrt(np.sum(positions_planetward**2, axis=1))
    LT_planetward = 6 / 90 * np.arctan2(positions_planetward[:, 1], positions_planetward[:, 0]) * 180 / np.pi + 12

#     R_tailward_occurence = np.histogram(R_tailward, bins=np.linspace(0, 100, 11))
#     R_planetward_occurence = np.histogram(R_planetward, bins=np.linspace(0, 100, 11))

#     LT_tailward_occurence = np.histogram(LT_tailward, bins=np.linspace(0, 24, 25))
#     LT_planetward_occurence = np.histogram(LT_planetward, bins=np.linspace(0, 24, 25))

    
#     print(np.diff(R_occurence[1]).shape)
#     print(R_tailward_occurence[0].shape)
#     print(R_occurence[0].shape)
#     print((R_tailward_occurence[0] / R_occurence[0] * 60).shape)
    
#     ax[0, 0].bar(R_occurence[1][:-1], R_occurence[0], 
#                  width=np.diff(R_occurence[1]), edgecolor='k', fill=False, align='edge')
    
#     ax[1, 0].bar(R_occurence[1][:-1], R_tailward_occurence[0], 
#                  width=np.diff(R_occurence[1]), edgecolor='r', fill=False, align='edge')
    
#     ax[2, 0].bar(R_occurence[1][:-1], R_planetward_occurence[0], 
#                  width=np.diff(R_occurence[1]), edgecolor='b', fill=False, align='edge')

#     ax[0, 1].bar(LT_occurence[1][:-1], LT_occurence[0], 
#                  width=np.diff(LT_occurence[1]), edgecolor='k', fill=False, align='edge')
    
#     ax[1, 1].bar(LT_occurence[1][:-1], LT_tailward_occurence[0], 
#                  width=np.diff(LT_occurence[1]), edgecolor='r', fill=False, align='edge')
    
#     ax[2, 1].bar(LT_occurence[1][:-1], LT_planetward_occurence[0], 
#                  width=np.diff(LT_occurence[1]), edgecolor='b', fill=False, align='edge')
          
    ax[0, 0].hist(R, bins=np.linspace(0, 110, 12), color='k', histtype='step')
    ax[1, 0].hist(R_tailward, bins=np.linspace(0, 100, 11), color='r', histtype='step')
    ax[2, 0].hist(R_planetward, bins=np.linspace(0, 100, 11), color='b', histtype='step')
    ax[0, 1].hist(LT, bins=np.linspace(0, 24, 25), color='k', histtype='step')
    ax[1, 1].hist(LT_tailward, bins=np.linspace(0, 24, 25), color='r', histtype='step')
    ax[2, 1].hist(LT_planetward, bins=np.linspace(0, 24, 25), color='b', histtype='step')
     
    
       
    #     ax[0, 1].bar(LT_occurence[1][:-1], LT_tailward_occurence[0] / LT_occurence[0] * 60, 
#                  width=np.diff(LT_occurence[1]), edgecolor='r', fill=False, align='edge')
    
#     ax[1, 1].bar(LT_occurence[1][:-1], np.ma.masked_invalid(LT_planetward_occurence[0] / LT_occurence[0] * 60), 
#                  width=np.diff(LT_occurence[1]), edgecolor='b', fill=False, align='edge')

    for a in ax[:, 1]:
        a.set_xlim([0, 6])
        a.set_xlabel('LT', weight='bold')
        a.label_outer()

    for a in ax[:, 0]:
        a.set_xlim([0, 110])
        a.set_xlabel('r ($R_J$)', weight='bold')
        a.label_outer()
    
    ax[0, 0].set_ylim([0, 6000])
    ax[0, 0].set_ylabel('Hours')
    ax[0, 0].text(0.05, 0.75, 'Juno Dwell-time', color='k', transform=ax[0, 0].transAxes)
    
    ax[0, 1].set_ylim([0, 6000])
    
    ax[1, 0].set_ylabel('#')
    ax[1, 0].text(0.05, 0.75, 'Tailward\nN={:d}'.format(len(R_tailward)), color='r', transform=ax[1, 0].transAxes)
    
    ax[1, 0].set_ylim([0, 20])
    ax[1, 1].set_ylim([0, 20])
    ax[2, 0].set_ylim([0, 15])
    ax[2, 1].set_ylim([0, 15])
    
    ax[1, 0].axvline(90, color='k', ls='dashed')
    ax[2, 0].axvline(90, color='k', ls='dashed')
    ax[1, 1].axvline(5, color='k', ls='dashed')
    ax[2, 1].axvline(5, color='k', ls='dashed')
    
    ax[2, 0].set_ylabel('#')
    ax[2, 0].text(0.05, 0.75, 'Planetward\nN={:d}'.format(len(R_planetward)), color='b', transform=ax[2, 0].transAxes)
    
    for i, a in enumerate(ax.flatten()):
        a.text(0.9, 0.87, '{:s})'.format(chr(ord('a') + i)), transform=a.transAxes, color='k')
        
    fig.tight_layout()
    # ax[0, 0].text(0.05, 0.75, 'Tailward\nN={:d}'.format(len(R_tailward)), color='r', transform=ax[0, 0].transAxes)
    # ax[1, 0].text(0.05, 0.75, 'Planetward\nN={:d}'.format(len(R_planetward)), color='b', transform=ax[1, 0].transAxes)
    
    fig.savefig('out_images/histogram_occurence_location.png', bbox_inches='tight')
    
plot_histograms()

In [ ]:
def plot_juno_trajectory():
    gs_kw = dict(height_ratios=[150, 75], hspace=0)
    fig, ax = plt.subplots(2, 1, sharex=True, dpi=300, gridspec_kw=gs_kw, figsize=(6, 8))

    ax1 = ax[0]
    ax2 = ax[1]
    
    ax1.plot(position[:, 0], position[:, 1], color='royalblue', zorder=1, alpha=0.5)
    ax2.plot(position[:, 0], position[:, 2], color='royalblue', zorder=1, alpha=0.5)
    # ax1.plot(position_masked[:, 0], position_masked[:, 1], color=[0.3, 0.3, 0.3], zorder=1)
    # ax2.plot(position_masked[:, 0], position_masked[:, 2], color=[0.3, 0.3, 0.3], zorder=1)

    ax1.plot(x_BS, y_BS, color='k', zorder=0)
    ax1.plot(x_MP, y_MP, color='k', ls='dashed', zorder=0)

    ax1.xaxis.set_minor_locator(AutoMinorLocator())
    ax1.yaxis.set_minor_locator(AutoMinorLocator())
    ax2.xaxis.set_minor_locator(AutoMinorLocator())
    ax2.yaxis.set_minor_locator(AutoMinorLocator())
    ax1.grid('on', which='minor', lw=0.5, ls='dotted')
    ax1.grid(color='k', ls='dashed', lw=0.25)
    ax2.grid('on', which='minor', lw=0.5, ls='dotted')
    ax2.grid(color='k', ls='dashed', lw=0.25)

    ax1.set_xlabel('X [R]', weight='bold')
    ax1.set_ylabel('Y [R]', weight='bold')
    ax2.set_xlabel('X [R]', weight='bold')
    ax2.set_ylabel('Z [R]', weight='bold')



    # ax1.set_xlim([-105, 40])
    ax1.set_ylim([-120, 30])

    ax2.set_xlim([-150, 40])
    ax2.set_ylim([-55, 20])

    # ax1.set_adjustable("datalim", share=True)

    ax1.set_aspect('equal', share=True)
    ax2.set_aspect('equal', share=True)

    ax1.arrow(20, -115, 10, 0, color='k', head_width=3)
    ax1.text(20, -110, 'To Sun')
    ax1.text(30, -60, 'MP')
    ax1.text(30, -90, 'BS')
    ax1.text(2, 1, 'Jupiter')

    # ax2.text(-60, -40, 'Juno Trajectory', color=[0.3, 0.3, 0.3], alpha=0.5, weight='bold')
    
    date_array = pd.date_range(
            start = dt.datetime(2016, 6, 20, 17, 30) + dt.timedelta(days=40),
            end = dt.datetime(2020, 5, 7, 23),
            freq='265D').to_pydatetime().tolist()
    
    juno_locations, _ = get_juno_position(date_array)
    
    
    print(juno_locations)
    for date, loc in zip(date_array, juno_locations):
        ax1.annotate(
            '{:s}'.format(date.strftime('%Y-%m-%d')),
            xy=(loc[0], loc[1] - 6),
            ha='center')
        
        ax2.annotate(
            '{:s}'.format(date.strftime('%Y-%m-%d')),
            xy=(loc[0], loc[2] - 6),
            ha='center')
        
    ax1.scatter(juno_locations[:, 0], juno_locations[:, 1], color='k', marker='*')
    ax2.scatter(juno_locations[:, 0], juno_locations[:, 2], color='k', marker='*')
    
    ax1.legend(['Juno Trajectory, Top-down view'])
    ax2.legend(['Juno Trajectory, Looking from dawn'])
    ax1.add_artist(plt.Circle((0, 0), 1.0, color='k', zorder=0))
    ax2.add_artist(plt.Circle((0, 0), 1.0, color='k', zorder=0))
    
    ax1.text(-0.15, 0.9, 'a)', bbox=dict(facecolor='w', edgecolor='k'), transform=ax1.transAxes)
    ax2.text(-0.15, 0.9, 'b)', bbox=dict(facecolor='w', edgecolor='k'), transform=ax2.transAxes)

    plt.subplots_adjust(wspace=0.3)
    fig.savefig('out_images/JunoTrajectory_20200727.png', bbox_inches='tight', facecolor='white')
    
plot_juno_trajectory()

In [ ]:
-1 % 4